# H Adatom on Si(100)

## 0. Introduction

This notebook builds the Si(100)2×1 slab with one H adatom on a dimer atom, the start and end structures of the H diffusion paths in the manuscript:

> **Hansen, U., & Vogl, P.**
> "Hydrogen passivation of silicon surfaces: A classical molecular-dynamics study."
> Physical Review B, 57(20), 13295–13304. (1998).
> [DOI: 10.1103/PhysRevB.57.13295](https://doi.org/10.1103/PhysRevB.57.13295)

The H adatom sits on the dangling bond of a dimer atom (Fig. 1). The start structure has it on the upper atom of a buckled dimer; each end structure has it where a path of Fig. 1 leads: the dimer partner for the intradimer path (a), the upper atom of the next dimer along the row for the intrarow path (b), the facing atom of the neighbouring row for the inter-row path (c). The structures are saved by name for the [simulation notebook](defect_point_adatom_silicon_hydrogen_SIMULATION.ipynb).

## 1. Prepare the Environment
### 1.1. Set up the parameters

In [ ]:
# Si(100)2x1 slab from Standata: eight layers, buckled dimers, a = 5.469 Å;
# the manuscript's dimers are symmetric (Sec. III A) and it states no lattice constant
MATERIAL_NAME = "mavrl-si-100-r"
SUPERCELL_MATRIX = [[2, 0, 0], [0, 2, 0], [0, 0, 1]]  # 64 Si
# SUPERCELL_MATRIX = [[3, 0, 0], [0, 3, 0], [0, 0, 1]]  # the manuscript's 144 Si (Sec. IV)

# Passivation of the top face; each material keeps one of its H
PASSIVANT = "H"
BOND_LENGTH = 1.46  # Si-H distance, in Angstroms
SURFACE = "top"
SHADOWING_RADIUS = 1.8  # in Angstroms, below the 2.30 Å dimer bond, so both dimer atoms get an H
DEPTH = 1.0  # in Angstroms, reaches the lower dimer atom, 0.65 Å below the upper one

# Site of the H in each material: Cartesian (x, y) of the Si atom it caps, in Angstroms;
# the dimers lie along x, the dimer rows along y
SITE_BY_MATERIAL_NAME = {
    "H on Si(100) start": [10.11, 0.0],  # upper atom of a dimer
    "H on Si(100) intradimer end": [12.32, 0.0],  # its dimer partner, Fig. 1 path (a)
    # "H on Si(100) intrarow end": [10.11, 3.87],  # upper atom of the next dimer along the row, path (b)
    # "H on Si(100) inter-row end": [4.59, 0.0],  # facing lower atom of the neighbouring row, path (c)
}

### 1.2. Install Packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples")

### 1.3. Get input material

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials

slab = Material.create(Materials.get_by_name_first_match(MATERIAL_NAME))

### 1.4. Create the supercell

In [ ]:
from mat3ra.made.tools.helpers import create_supercell
from mat3ra.made.tools.modify import translate_to_z_level

supercell = create_supercell(slab, supercell_matrix=SUPERCELL_MATRIX)
supercell = translate_to_z_level(supercell, "center")

## 2. Create the H Adatom
### 2.1. Passivate the top face

In [ ]:
from mat3ra.made.tools.helpers import passivate_surface

passivated_slab = passivate_surface(
    material=supercell,
    passivant=PASSIVANT,
    bond_length=BOND_LENGTH,
    surface=SURFACE,
    shadowing_radius=SHADOWING_RADIUS,
    depth=DEPTH,
)

### 2.2. Keep one H per material
Each material keeps the slab and the H on its site: the topmost atom within 1 Å of the site, seen from above.

In [ ]:
from mat3ra.made.tools.analyze.other import get_local_extremum_atom_index
from mat3ra.made.tools.modify import filter_by_ids

materials = []
for name, site in SITE_BY_MATERIAL_NAME.items():
    hydrogen_id = get_local_extremum_atom_index(passivated_slab, site, use_cartesian_coordinates=True)
    material = filter_by_ids(passivated_slab, [*supercell.basis.elements.ids, hydrogen_id])
    material.name = name
    materials.append(material)

## 3. Visualize the Slabs with the H Adatom

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

visualize([{"material": material, "title": material.name} for material in materials], viewer="wave")

## 4. Pass data to the outside runtime

In [ ]:
from mat3ra.notebooks_utils.io import download_content_to_file
from mat3ra.notebooks_utils.material import set_materials

set_materials(materials)
for material in materials:
    download_content_to_file(material.to_json(), f"{material.name}.json")

## References

[1] U. Hansen and P. Vogl, "Hydrogen passivation of silicon surfaces: A classical molecular-dynamics study", Phys. Rev. B 57, 13295 (1998). [DOI: 10.1103/PhysRevB.57.13295](https://doi.org/10.1103/PhysRevB.57.13295)